# 🧳 Personal Finance / Budget Agent — "Can I afford this trip?"
### Building a GenAI Agent from scratch — Fundamentals → Advanced Agentic AI

**Session goal:** By the end of this notebook, you will have built a real agent, from a single naive
prompt all the way up to a bounded, tool-using, constraint-solving loop — the same pattern used in
production agents (order processing, resource allocation, scheduling).

**What we will build, step by step:**

| Step | What we add | Concept |
|---|---|---|
| 1 | A single naive prompt | Baseline — and where LLMs go wrong |
| 2 | Forced structured (JSON) output | Structured Output |
| 3 | A deterministic calculator tool | Tool Use / Function Calling |
| 4 | A feasibility-check + recommendation loop | Constraint Satisfaction + Bounded ReAct Loop |
| 5 | Put it all together | The full Budget Agent |

We use **OpenRouter** as our LLM provider — it gives every student a way to call many models
through **one single API key**, using the same `openai` Python SDK, with **no payment method
required**. This lab sticks entirely to OpenRouter's **free-tier (`:free`) models** — no credit
card, no billing setup, just a signup. The code you write today will work with almost any model on
OpenRouter, free or paid, by changing one line.


## 🔑 Step 0 — Get your OpenRouter API key

1. Go to **[openrouter.ai](https://openrouter.ai)** and sign up (Google/GitHub login is fine).
2. Go to **[openrouter.ai/keys](https://openrouter.ai/keys)** → click **"Create Key"**.
3. Copy the key (it looks like `sk-or-v1-xxxxxxxx...`).
4. In Colab, click the **🔑 key icon** in the left sidebar → **Secrets** → add a new secret named
   `OPENROUTER_API_KEY` and paste your key as the value. Toggle **"Notebook access"** ON.

> Why OpenRouter and not a direct provider key? One key → many models. You can swap one model id
> for another by changing **one line**, with zero other code changes. Great for comparing how
> different models behave as agents.

**⚠️ Free-tier models only — this matters:** we're assuming you have no payment method on file, so
this lab only uses models with a `:free` id suffix (0 cost). Model naming looks similar to paid
models (e.g. `anthropic/claude-3.5-haiku` is **not** free, but `google/gemma-4-31b-it:free` is) —
always double-check on [openrouter.ai/models](https://openrouter.ai/models?max_price=0) before
picking one, since which models are offered for free changes over time. As of writing, these free
models support the tool/function calling this lab needs:
- `google/gemma-4-31b-it:free`
- `qwen/qwen3.8-27b:free`
- `nvidia/nemotron-3-super-120b-a12b:free`

We default to the first one below — feel free to swap `MODEL` for another `:free` one and compare.

**Rate limits (important for a classroom setting):** an account that has never purchased credits
is capped at **20 requests/minute and 50 requests/day per key** on `:free` models. This notebook
makes several calls per run (intent extraction + up to a few loop iterations), so if a cell errors
with a 429, that's the daily/per-minute cap, not a bug — wait a bit or pace yourself while
experimenting in the class exercises.

**A second, different 429 you may see:** `"<model> is temporarily rate-limited upstream"`. This
one isn't about *your* key at all — `:free` models run on a shared pool of the underlying
provider's spare capacity, and that shared pool can get saturated by everyone's traffic at once.
The fix isn't to wait on your own quota; it's to fall back to another free model. That's exactly
what `FALLBACK_MODELS` in the next cell does automatically — no code changes needed if you hit
this.


In [ ]:
# Install dependencies (OpenRouter is OpenAI-API-compatible, so we just use the `openai` SDK)
!pip install openai --quiet
print("✅ Installed.")


In [ ]:
import os
from google.colab import userdata

# Pull the key from Colab Secrets (recommended) — falls back to manual input if not set
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not OPENROUTER_API_KEY:
        raise ValueError("empty")
except Exception:
    OPENROUTER_API_KEY = input("Paste your OpenRouter API key: ").strip()

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Key loaded." if OPENROUTER_API_KEY else "❌ No key found.")


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# Free-tier model (":free" suffix = $0 cost). Swap for another ":free" model id to compare —
# see the model list in Step 0 for current tool-calling-capable free options.
MODEL = "google/gemma-4-31b-it:free"

# ":free" models run on a shared pool of upstream provider capacity — under load you can get a
# 429 ("temporarily rate-limited upstream") even though YOUR key hasn't hit its own daily/minute
# cap. OpenRouter can auto-retry the next model in this list when that happens — pass it via
# extra_body so no manual retry loop is needed. All three support the tool calling this lab needs.
FALLBACK_MODELS = ["qwen/qwen3.8-27b:free", "nvidia/nemotron-3-super-120b-a12b:free"]

def ask(messages, tools=None, tool_choice=None, **kwargs):
    """Thin wrapper around the chat completions call so every step below stays short."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice=tool_choice,
        extra_body={"models": [MODEL] + FALLBACK_MODELS},
        **kwargs,
    )
    return resp

# 👋 Hello World — confirms the key, base_url, and model are all wired up correctly
# before we build anything on top of them.
test = ask([{"role": "user", "content": "Say 'Hello World, agent lab is ready!' and nothing else."}])
print(test.choices[0].message.content)



---
## Step 1 — The naive baseline (and where it breaks)

Let's just... ask the question directly. No structure, no tools, no constraints. This is what most
people think "using AI" means. Watch what happens.


In [ ]:
naive_prompt = """
I get a monthly stipend of ₹8000. My fixed monthly costs (rent + food) are ₹5000.
I want to take a 3-day trip to Goa. Can I afford it? Give me a straight answer.
"""

response = ask([{"role": "user", "content": naive_prompt}])
print(response.choices[0].message.content)



### 🔍 What just happened — discuss in class

Run the cell above a few times, or try changing the destination. You'll typically see one or more
of these problems:

- **Hallucinated numbers.** The model invents a "typical Goa trip cost" from its training data,
  which may be outdated or just wrong — it did **not** look anything up.
- **Inconsistent formatting.** Sometimes prose, sometimes a bulleted list, sometimes a table.
  Your code (or a UI) can't reliably parse *any* of these.
- **Sycophancy.** Models are tuned to be agreeable. Ask it "I really want to go, can I afford it?"
  and notice it leans towards "sure, you can probably manage it!" even when the math doesn't work.
- **No real arithmetic.** It's *predicting plausible-sounding numbers*, not calculating them.

This is the motivating failure case for everything we build next.



---
## Step 2 — Forcing structured output

An **agent** is only useful if the *rest of your system* (a UI, a database, another function) can
reliably act on what it returns. Free text is not that. We fix this by defining a strict JSON
**schema** and forcing the model to fill it in — via a "tool" the model is required to call.

This is the same mechanism as real tool/function calling — we're just using it here to **force a
shape**, not to run real code yet. That comes in Step 3.


In [ ]:
import json

# The schema we want the model to ALWAYS return, no matter how the question is phrased
intent_schema = {
    "type": "function",
    "function": {
        "name": "extract_trip_request",
        "description": "Extract structured trip-affordability parameters from the user's message.",
        "parameters": {
            "type": "object",
            "properties": {
                "monthly_income": {"type": "number", "description": "Total monthly stipend/income in INR"},
                "fixed_monthly_costs": {"type": "number", "description": "Rent + food + other fixed costs in INR"},
                "destination": {"type": "string", "description": "Trip destination"},
                "trip_duration_days": {"type": "integer", "description": "Length of the trip in days"},
                "hard_constraints": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Non-negotiable preferences the user stated, e.g. 'must stay at a resort'",
                },
            },
            "required": ["monthly_income", "fixed_monthly_costs", "destination", "trip_duration_days"],
        },
    },
}

def extract_intent(user_message: str) -> dict:
    resp = ask(
        [{"role": "user", "content": user_message}],
        tools=[intent_schema],
        tool_choice={"type": "function", "function": {"name": "extract_trip_request"}},
    )
    call = resp.choices[0].message.tool_calls[0]
    return json.loads(call.function.arguments)

intent = extract_intent(naive_prompt)
print(json.dumps(intent, indent=2))



### 🔍 Why this matters

Try feeding it a messier sentence — mix up the order of information, add irrelevant details, or
even write in Hindi/Tanglish mixed with English. The **schema stays identical**. This is the core
idea of structured output: *the shape of the answer is a contract, not a suggestion.*

Try it:


In [ ]:
messy_message = "yaar so basically I earn 8k a month, rent+food eats up 5000 of that, thinking of a 3 day Goa trip, non-negotiable I want AC room only"
print(json.dumps(extract_intent(messy_message), indent=2))



---
## Step 3 — Giving the agent a real calculator (and a real cost lookup)

LLMs are notoriously bad at arithmetic — they're predicting tokens, not computing. The fix is
simple and important: **any deterministic computation should be a real function call, never left
to the model's "reasoning."** The model's job is to decide *when* to call a tool and *how to use*
the result — not to do the math itself.

We'll give the agent two tools:
1. `estimate_trip_cost` — a (simulated) cost lookup, standing in for a real web-search/pricing API
2. `calculator` — basic arithmetic, so the model never invents a sum

> In a real deployment, `estimate_trip_cost` would call a travel API or web search. We simulate it
> here with a small lookup table so the lab works offline and deterministically — swap in a real
> API call as a stretch goal.


In [ ]:
# --- Simulated cost data (stand-in for a real travel-price API / web search tool) ---
DESTINATION_COSTS = {
    "goa":      {"travel_per_day": 600, "stay_per_day": 500, "food_per_day": 300},
    "manali":   {"travel_per_day": 900, "stay_per_day": 400, "food_per_day": 250},
    "pondicherry": {"travel_per_day": 400, "stay_per_day": 450, "food_per_day": 280},
    "default":  {"travel_per_day": 700, "stay_per_day": 500, "food_per_day": 300},
}

def estimate_trip_cost(destination: str, trip_duration_days: int) -> dict:
    """Deterministic tool: looks up per-day costs and multiplies by duration.
    This stands in for a real pricing/search API."""
    key = destination.strip().lower()
    rates = DESTINATION_COSTS.get(key, DESTINATION_COSTS["default"])
    breakdown = {k.replace("_per_day", ""): v * trip_duration_days for k, v in rates.items()}
    breakdown["total"] = sum(breakdown.values())
    return breakdown

def calculator(expression: str) -> float:
    """Deterministic tool: safely evaluates a basic arithmetic expression."""
    allowed = "0123456789+-*/(). "
    if not all(ch in allowed for ch in expression):
        raise ValueError(f"Unsafe expression: {expression}")
    return eval(expression)  # safe here: input is character-whitelisted above

# Quick test
print(estimate_trip_cost("Goa", 3))
print(calculator("(600+500+300)*3"))



### 🔍 Demo: prove the model gets arithmetic wrong on its own

Let's deliberately compare the model doing the sum **itself** vs. calling our `calculator` tool.


In [ ]:
# Ask the model to do the arithmetic in free text (no tool) — watch for errors, especially
# with slightly awkward numbers.
trap_prompt = "What is 3897 * 4 + 1256 - 389? Just give me the final number, nothing else."
print("Model's own arithmetic:", ask([{"role": "user", "content": trap_prompt}]).choices[0].message.content)
print("Actual answer (Python): ", 3897 * 4 + 1256 - 389)



Run that a few times with different numbers — the model is often close but wrong, especially as
numbers get bigger or the expression gets more steps. **This is exactly why Step 4's feasibility
check must never be judged by the model itself — it has to be a real comparison in code.**



---
## Step 4 — The constraint-satisfaction loop (the heart of this agent)

Now the real agentic part. Given the extracted intent and the estimated cost, we:

1. **Check feasibility** — `available_budget = monthly_income - fixed_monthly_costs`. Is
   `estimated_cost <= available_budget`? (Plain Python comparison — never ask the model this.)
2. If **not feasible**, ask the model to propose **one trade-off recommendation** that would
   reduce cost (cheaper stay, fewer days, bus instead of flight, etc.), grounded in the actual
   cost breakdown.
3. **Apply the recommendation deterministically**, recompute the cost, and check feasibility again.
4. Repeat until feasible **or** we hit a maximum number of iterations (a **bounded loop** — real
   agents must never be allowed to loop forever).

This is a genuine bounded ReAct loop: **Thought → Action → Observation → repeat**, with a real
business objective instead of a toy example.


In [ ]:
recommendation_schema = {
    "type": "function",
    "function": {
        "name": "propose_savings",
        "description": "Propose exactly one concrete change that reduces trip cost, given a cost breakdown and any hard constraints that must not be violated.",
        "parameters": {
            "type": "object",
            "properties": {
                "change": {"type": "string", "description": "A short, concrete description of the change, e.g. 'Switch stay from hotel to hostel'"},
                "category": {"type": "string", "enum": ["travel", "stay", "food"], "description": "Which cost category this change reduces"},
                "estimated_new_daily_cost": {"type": "number", "description": "The new estimated per-day cost for that category after the change"},
            },
            "required": ["change", "category", "estimated_new_daily_cost"],
        },
    },
}

def propose_savings(breakdown: dict, hard_constraints: list, duration: int) -> dict:
    daily_costs = {k: v // duration for k, v in breakdown.items() if k != "total"}
    prompt = (
        f"Current daily cost breakdown for a {duration}-day trip (per day, in INR):\n"
        f"{json.dumps(daily_costs, indent=2)}\n\n"
        f"Hard constraints that must NOT be violated: {hard_constraints or 'None'}\n\n"
        "Propose exactly ONE realistic change to reduce cost. Do not violate any hard constraint."
    )
    resp = ask(
        [{"role": "user", "content": prompt}],
        tools=[recommendation_schema],
        tool_choice={"type": "function", "function": {"name": "propose_savings"}},
    )
    call = resp.choices[0].message.tool_calls[0]
    return json.loads(call.function.arguments)


def solve_budget(intent: dict, max_iterations: int = 4) -> dict:
    """The bounded constraint-satisfaction loop."""
    available_budget = intent["monthly_income"] - intent["fixed_monthly_costs"]
    duration = intent["trip_duration_days"]
    destination = intent["destination"]
    hard_constraints = intent.get("hard_constraints", [])

    breakdown = estimate_trip_cost(destination, duration)
    recommendations_applied = []

    for iteration in range(1, max_iterations + 1):
        feasible = breakdown["total"] <= available_budget   # <-- real comparison, not model judgment

        if feasible:
            return {
                "feasible": True,
                "available_budget": available_budget,
                "estimated_cost": breakdown["total"],
                "shortfall": 0,
                "breakdown": breakdown,
                "recommendations_applied": recommendations_applied,
                "iterations_used": iteration,
                "verdict": "Affordable" if not recommendations_applied else "Affordable after adjustments",
            }

        # Not feasible yet — and we've run out of iterations
        if iteration == max_iterations:
            break

        # Ask the model for ONE grounded recommendation, then apply it deterministically
        rec = propose_savings(breakdown, hard_constraints, duration)
        category = rec["category"]
        new_daily = rec["estimated_new_daily_cost"]
        old_total_for_category = breakdown[category]
        breakdown[category] = new_daily * duration
        breakdown["total"] = sum(v for k, v in breakdown.items() if k != "total")
        recommendations_applied.append({
            "change": rec["change"],
            "savings": old_total_for_category - breakdown[category],
        })

    # Exhausted iterations without becoming feasible
    return {
        "feasible": False,
        "available_budget": available_budget,
        "estimated_cost": breakdown["total"],
        "shortfall": breakdown["total"] - available_budget,
        "breakdown": breakdown,
        "recommendations_applied": recommendations_applied,
        "iterations_used": max_iterations,
        "verdict": "Not affordable even after adjustments — consider a shorter or cheaper trip",
    }



### 🔍 Try it

Run the full loop on our original example and watch the trace.


In [ ]:
result = solve_budget(intent, max_iterations=4)
print(json.dumps(result, indent=2))



Notice:
- The **feasibility check is a plain Python `<=` comparison** — never delegated to the model.
- Each recommendation is generated by the model (creative judgment) but **applied deterministically**
  in code (arithmetic).
- The loop has a **hard ceiling** (`max_iterations`). Try setting it to `1` and see the agent
  correctly report "not affordable" instead of looping forever trying to force a fit.



---
## Step 5 — Putting it all together: the full Budget Agent

Now we wire Steps 2–4 into a single function that takes a raw, messy user sentence and returns a
clean structured verdict — exactly the schema we sketched out at the start of the session.


In [ ]:
def budget_agent(user_message: str, max_iterations: int = 4) -> dict:
    print("🧠 Step A — Extracting structured intent from your message...")
    intent = extract_intent(user_message)
    print(json.dumps(intent, indent=2))

    print("\n💰 Step B — Estimating cost and solving constraints...")
    result = solve_budget(intent, max_iterations=max_iterations)

    print("\n✅ Final structured result:")
    print(json.dumps(result, indent=2))
    return result


In [ ]:
# Try your own scenario here!
my_message = """
I earn a 9000 rupee stipend. Rent and food take up 6000. I want to go to Manali for 4 days,
and I absolutely need an AC room — non-negotiable.
"""

final_result = budget_agent(my_message)


---
## 🧪 Try it yourself (class exercise)

Edit the cell above (or make a new cell) and try:

1. A trip that's clearly affordable — does the agent correctly skip straight to "Affordable" in
   iteration 1 without inventing unnecessary recommendations?
2. A trip that's clearly *not* affordable even with adjustments (e.g. huge duration, tiny budget) —
   does it correctly say "Not affordable" instead of hallucinating a way to make it fit?
3. Add a hard constraint the model should never violate (e.g. "must fly, not take a bus") —
   check that `propose_savings` respects it.
4. Set `max_iterations=1` on an infeasible case — confirm the agent stops immediately with an
   honest "not affordable" instead of pretending to solve it.
5. **Stretch:** swap `MODEL` at the top of the notebook to a different **free-tier** OpenRouter
   model (e.g. `"qwen/qwen3.8-27b:free"` or `"nvidia/nemotron-3-super-120b-a12b:free"`) and re-run
   everything — does the behavior change? (Stick to `:free` models — see Step 0 for why.)



---
## 🧵 Recap — what you actually built

| Layer | What it means in production systems |
|---|---|
| Structured output (Step 2) | Contract between the LLM and the rest of your codebase — no more regex-parsing prose |
| Tool use (Step 3) | Delegate anything deterministic (math, lookups, DB calls) — never trust the model's own arithmetic |
| Constraint loop (Step 4) | Real agents solve for a goal under limits, and must have a **bounded** stopping condition |
| Full agent (Step 5) | Orchestration: the model handles judgment calls, your code handles ground truth |

**The one-line takeaway to leave students with:**
> An agent is not a smarter chatbot — it's a *loop* around an LLM, where the LLM makes judgment
> calls and your code enforces ground truth, structure, and stopping conditions.
